In [1]:
import pulp as lp
import pandas as pd

T = 8
A0 = 20600000

instruments = {
    "DTF3": {"d": 1, "g": 0.0836, "tax_rate": 0.04},   # 3 meses
    "DTF6": {"d": 2, "g": 0.0845, "tax_rate": 0.04},   # 4 meses
    "DTF9": {"d": 3, "g": 0.0854, "tax_rate": 0.07},   # 9 meses
    "DTF12": {"d": 4, "g": 0.0863, "tax_rate": 0.07},  # 12 meses
}

for instrument in instruments:
    g_bruto = instruments[instrument]["g"]
    tax_rate = instruments[instrument]["tax_rate"]
    instruments[instrument]["g_net"] = g_bruto * (1 - tax_rate)

for i in instruments:
    for t in range(1, T+1):
        vence_en = t + instruments[i]["d"]
        puede_usarse = vence_en <= T + 1

S = ["DTF3", "DTF6"]
L = ["DTF9", "DTF12"]

pS_min, pS_max = 0.25, 0.6
pL_min, pL_max = 0.4, 0.75

model = lp.LpProblem("Investment_Reinvestment_Model_with_Tax", lp.LpMaximize)

x = lp.LpVariable.dicts("x", [(i, t) for i in instruments for t in range(1, T+1)], lowBound=0)
A = lp.LpVariable.dicts("A", range(1, T+2), lowBound=0)

# Funcion Objetivo
model += A[T+1] - A0, "Maximize_net_final_capital"

#Restricciones 

model += A[T+1] >= A0, "No_loss_condition"

model += A[1] == A0, "Initial_capital"

# Vencimiento
for t in range(1, T+1):
    for i in instruments:
        if t + instruments[i]["d"] > T + 1:
            model += x[(i, t)] == 0, f"No_mature_{i}_t{t}"

# Capacidad
for t in range(1, T+1):
    model += lp.lpSum(x[(i, t)] for i in instruments) <= A[t], f"Capacity_t{t}"

for t in range(1, T+1):
    matured_net = lp.lpSum(
        x[(i, t - instruments[i]["d"])] * (1 + instruments[i]["g_net"])
        for i in instruments if t - instruments[i]["d"] >= 1
    )
    
    model += A[t+1] == A[t] - lp.lpSum(x[(i, t)] for i in instruments) + matured_net, f"Evolution_t{t}"

for t in range(1, T+1):
    available_S = [i for i in S if t + instruments[i]["d"] <= T + 1]
    available_L = [i for i in L if t + instruments[i]["d"] <= T + 1]
    
    if available_S:
        model += lp.lpSum(x[(i, t)] for i in available_S) >= pS_min * A[t], f"ShortMin_t{t}"
        model += lp.lpSum(x[(i, t)] for i in available_S) <= pS_max * A[t], f"ShortMax_t{t}"
    
    if available_L:
        model += lp.lpSum(x[(i, t)] for i in available_L) >= pL_min * A[t], f"LongMin_t{t}"
        model += lp.lpSum(x[(i, t)] for i in available_L) <= pL_max * A[t], f"LongMax_t{t}"

model.solve(lp.PULP_CBC_CMD(msg=0))

print("RESULTADOS OPTIMIZADOS CON RETENCIÓN EN LA FUENTE")
print("Estado:", lp.LpStatus[model.status])
print(f"Capital inicial: ${A0:,.2f}")
print(f"Capital final (después de impuestos): ${A[T+1].value():,.2f}")
print(f"Retorno total neto: ${A[T+1].value() - A0:,.2f}")
print(f"Retorno porcentual neto: {((A[T+1].value() - A0) / A0 * 100):.2f}%")

total_taxes = 0
tax_details = []

for t in range(1, T+1):
    for i in instruments:
        # Solo calcular impuestos para inversiones que realmente se hicieron y que ya vencieron
        if (t - instruments[i]["d"] >= 1 and 
            x[(i, t - instruments[i]["d"])].value() is not None and 
            x[(i, t - instruments[i]["d"])].value() > 0):
            
            tax_amount = x[(i, t - instruments[i]["d"])].value() * instruments[i]["g"] * instruments[i]["tax_rate"]
            total_taxes += tax_amount
            tax_details.append({
                "period": t,
                "instrument": i,
                "investment": x[(i, t - instruments[i]["d"])].value(),
                "tax_paid": tax_amount
            })

capital_teorico_sin_impuestos = A0
for t in range(1, T+1):
    for i in instruments:
        if (t - instruments[i]["d"] >= 1 and 
            x[(i, t - instruments[i]["d"])].value() is not None and 
            x[(i, t - instruments[i]["d"])].value() > 0):
            
            capital_teorico_sin_impuestos += x[(i, t - instruments[i]["d"])].value() * instruments[i]["g"]

print(f"Capital final teórico sin impuestos: ${capital_teorico_sin_impuestos:,.2f}")
print(f"Impacto total de impuestos: -${capital_teorico_sin_impuestos - A[T+1].value():,.2f}")

# Inversiones
print("\nINVERSIONES REALIZADAS:")
investment_data = []
for t in range(1, T+1):
    for i in instruments:
        if x[(i, t)].value() is not None and x[(i, t)].value() > 1:
            inv_value = x[(i, t)].value()
            vence_en = t + instruments[i]["d"]
            retorno_bruto = inv_value * instruments[i]["g"]
            retorno_neto = inv_value * instruments[i]["g_net"]
            
            investment_data.append({
                "Periodo": t,
                "Instrumento": i,
                "Inversión": f"${inv_value:,.2f}",
                "Vence en": vence_en,
                "Retorno Bruto": f"${retorno_bruto:,.2f}",
                "Retorno Neto": f"${retorno_neto:,.2f}",
                "Tasa Imp": f"{instruments[i]['tax_rate']:.1%}"
            })

if investment_data:
    df_inv = pd.DataFrame(investment_data)
    print(df_inv.to_string(index=False))
else:
    print("No se realizaron inversiones significativas")

print("\nEVOLUCIÓN DEL CAPITAL:")
for t in range(1, T+2):
    if A[t].value() is not None:
        print(f"Periodo {t}: ${A[t].value():,.2f}")


RESULTADOS OPTIMIZADOS CON RETENCIÓN EN LA FUENTE
Estado: Optimal
Capital inicial: $20,600,000.00
Capital final (después de impuestos): $24,166,318.00
Retorno total neto: $3,566,318.00
Retorno porcentual neto: 17.31%
Capital final teórico sin impuestos: $25,480,470.38
Impacto total de impuestos: -$1,314,152.38

INVERSIONES REALIZADAS:
 Periodo Instrumento      Inversión  Vence en Retorno Bruto  Retorno Neto Tasa Imp
       1        DTF3 $12,360,000.00         2 $1,033,296.00   $991,964.16     4.0%
       1        DTF9  $8,240,000.00         4   $703,696.00   $654,437.28     7.0%
       3        DTF3  $8,011,178.50         4   $669,734.52   $642,945.14     4.0%
       3        DTF9  $5,340,785.70         6   $456,103.10   $424,175.88     7.0%
       5        DTF3  $4,387,140.20         6   $366,764.92   $352,094.32     4.0%
       5        DTF9 $13,161,421.00         8 $1,123,985.35 $1,045,306.38     7.0%
       7        DTF3  $6,302,517.70         8   $526,890.48   $505,814.86     4.0%